# nb_02 — Silver: conform, deduplicate, validate, convert

One row per workforce event.

1. **Deduplicate** replayed `event_id`s — keep the latest `ingest_ts`.
2. **Conform** the work-country code — `CORE_HR` emits ISO-3, `PAYROLL` emits ISO-2.
3. **Validate & quarantine** — negative pay, orphan employees, bad dates, unknown currency.
4. **Convert** pay amounts from local currency to **CAD** (the grid is in CAD).
5. Write `silver.workforce_event`.

> Note: many events (leaves, deployments) legitimately carry **no amount**. We
> quarantine *negative* pay, not null pay.

## Load Bronze inputs

**Summary.** Creates the `silver` schema and loads the Bronze tables this notebook conforms: raw events (including replays) plus the workers, workers-delta, and FX reference tables.

<details>
<summary>Line-by-line details</summary>

- `from pyspark.sql import functions as F, Window as W` — the helpers used for dedup and conforming.
- `CREATE SCHEMA IF NOT EXISTS silver` — ensure the Silver schema exists.
- `raw`, `wk`, `wkd`, `fx` — references to `bronze.workforce_events_raw`, `bronze.workers`, `bronze.workers_delta`, and `bronze.fx_rates`.
- `print(...)` — report the raw event count (still including replays).

</details>

In [ ]:
from pyspark.sql import functions as F, Window as W
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")
raw   = spark.table("bronze.workforce_events_raw")
wk    = spark.table("bronze.workers")
wkd   = spark.table("bronze.workers_delta")
fx    = spark.table("bronze.fx_rates")
print(f"bronze events (raw, incl replays): {raw.count():,}")

## 1. Deduplicate replays

**Summary.** Keeps only the newest copy of each `event_id`, removing the payroll system's replayed duplicates.

<details>
<summary>Line-by-line details</summary>

- `W.partitionBy("event_id").orderBy(F.col("ingest_ts").desc())` — a window per event, newest ingest first.
- `withColumn("_rn", F.row_number().over(w)).filter("_rn=1").drop("_rn")` — number rows within each event and keep only rank 1 (the latest).
- `print(...)` — report the surviving count and how many duplicates were removed.

</details>

In [ ]:
w = W.partitionBy("event_id").orderBy(F.col("ingest_ts").desc())

dedup = raw.withColumn("_rn", F.row_number().over(w)).filter("_rn=1").drop("_rn")

print(f"after dedup: {dedup.count():,}  (removed {raw.count()-dedup.count():,})")

## 2. Conform the work-country code to ISO-3

**Summary.** Normalizes the work-country code so both source systems agree: `CORE_HR` already sends ISO-3, `PAYROLL` sends ISO-2, which a small static map converts.

<details>
<summary>Line-by-line details</summary>

- `spark.createDataFrame([...], ["_i2","_i3"])` — a tiny ISO-2 to ISO-3 reference for the countries in scope.
- `withColumn("wcc3", F.when(F.length(...)==3, ...))` — keep values that are already ISO-3.
- `.join(iso, dedup.work_country_code==iso._i2, "left")` — look up the ISO-3 for ISO-2 values.
- `withColumn("work_country_iso3", F.coalesce("wcc3","_i3"))` — prefer the already-ISO-3 value, else the mapped one; helper columns are dropped.

</details>

In [ ]:
iso = spark.createDataFrame([
    ("CA","CAN"),("US","USA"),("GB","GBR"),("FR","FRA"),("DE","DEU"),("JP","JPN"),
    ("AU","AUS"),("SG","SGP"),("BR","BRA"),("AE","ARE"),("KE","KEN"),("IN","IND"),
    ("MX","MEX")], ["_i2","_i3"])

conf = (dedup
    .withColumn("wcc3", F.when(F.length("work_country_code")==3, F.col("work_country_code")))
    .join(iso, dedup.work_country_code==iso._i2, "left")
    .withColumn("work_country_iso3", F.coalesce("wcc3","_i3"))
    .drop("_i2","_i3","wcc3"))

## 3. Validate & quarantine

| Rule | Reason |
|------|--------|
| `amount_local < 0` | `negative_pay` |
| `employee_id` doesn't resolve | `orphan_employee` |
| `event_date` before 2021-01-01 or future | `date_out_of_range` |
| unknown currency **or** unresolved country | `unresolved_reference` |

Null amounts are **valid** (non-pay events). Failing rows → quarantine.

**Summary.** Flags rows that break data-quality rules (negative pay, orphan employees, out-of-range dates, unresolved currency/country), writes the failures to a quarantine table, and keeps the clean rows.

<details>
<summary>Line-by-line details</summary>

- `valid_emp` / `valid_ccy` — sets of known employee IDs (workers plus workers-delta) and known currencies, collected once.
- `F.when(...).when(...)` chain — assigns the first matching `dq_reason`: `negative_pay`, `orphan_employee`, `date_out_of_range`, or `unresolved_reference`. Null amounts are treated as valid.
- `quarantine = flagged.filter("dq_reason IS NOT NULL")` and `clean = flagged.filter("dq_reason IS NULL").drop("dq_reason")` — split failures from clean rows.
- `quarantine.write ... saveAsTable("silver.workforce_event_quarantine")` — persist the failures for review.
- The `groupBy("dq_reason").count()` and `print` summarize what was quarantined and how many rows continue.

</details>

In [ ]:
valid_emp = set(r.employee_id for r in
    wk.select("employee_id").union(wkd.select("employee_id")).distinct().collect())

valid_ccy = set(r.currency for r in fx.select("currency").distinct().collect())

flagged = (conf.withColumn("dq_reason",
    F.when(F.col("amount_local") < 0, "negative_pay")
     .when(~F.col("employee_id").isin(list(valid_emp)), "orphan_employee")
     .when((F.col("event_date") < F.lit("2021-01-01")) |
           (F.col("event_date") > F.current_date()), "date_out_of_range")
     .when(~F.col("local_currency").isin(list(valid_ccy)) |
           F.col("work_country_iso3").isNull(), "unresolved_reference")))

quarantine = flagged.filter("dq_reason IS NOT NULL")

clean = flagged.filter("dq_reason IS NULL").drop("dq_reason")

(quarantine.write.format("delta").mode("overwrite").option("overwriteSchema","true")
    .saveAsTable("silver.workforce_event_quarantine"))

quarantine.groupBy("dq_reason").count().orderBy("dq_reason").show()

print(f"clean rows continuing: {clean.count():,}")

## 4. Convert pay to CAD

The pay grid is in CAD, but international offices pay in local currency. Convert
`amount_local` → `amount_cad` on `(rate_month, currency)`. Non-pay events keep a
null amount.

**Summary.** Converts local-currency amounts to CAD using the FX rate for the event month, selects the conformed columns, and writes the trustworthy `silver.workforce_event` table.

<details>
<summary>Line-by-line details</summary>

- `fx_lkp` — the FX table aliased to `_rm` (rate month), `_ccy` (currency), and `cad_per_unit`.
- `withColumn("rate_month", F.date_format("event_date","yyyy-MM"))` — the join key month.
- `.join(fx_lkp, (rate_month==_rm) & (local_currency==_ccy), "left")` — attach the matching rate.
- `withColumn("amount_cad", F.when(amount_local isNull, None).otherwise(round(amount_local * coalesce(cad_per_unit, 1.0), 2)))` — null stays null; otherwise convert and round (missing rate defaults to 1.0).
- The final `.select(...)` casts and orders the output columns (`amount_local`/`amount_cad` as `decimal(18,2)`).
- `write ... saveAsTable("silver.workforce_event")` and the `print`/`show` confirm the result.

</details>

In [ ]:
fx_lkp = fx.select(F.col("rate_month").alias("_rm"),
                   F.col("currency").alias("_ccy"), "cad_per_unit")
silver = (clean
    .withColumn("rate_month", F.date_format("event_date","yyyy-MM"))
    .join(fx_lkp, (F.col("rate_month")==F.col("_rm")) &
                  (F.col("local_currency")==F.col("_ccy")), "left")
    .withColumn("amount_cad",
        F.when(F.col("amount_local").isNull(), None)
         .otherwise(F.round(F.col("amount_local")*F.coalesce("cad_per_unit",F.lit(1.0)),2)))
    .select("event_id", F.to_date("event_date").alias("event_date"),
            "employee_id","cost_center_id",
            "classification_group", F.col("classification_level").cast("int").alias("classification_level"),
            "event_type",
            F.col("amount_local").cast("decimal(18,2)").alias("amount_local"),
            "local_currency", "work_country_iso3",
            F.col("amount_cad").cast("decimal(18,2)").alias("amount_cad"),
            "source_system", F.to_timestamp("ingest_ts").alias("ingest_ts")))

(silver.write.format("delta").mode("overwrite").option("overwriteSchema","true")
    .saveAsTable("silver.workforce_event"))

print(f"silver.workforce_event: {silver.count():,} rows")

silver.show(5, truncate=False)